In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# %%capture

# !pip install "unsloth[colab-new]"

In [6]:
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA: 12.8


In [7]:
from unsloth import FastLanguageModel

print("Unsloth loaded successfully")

Unsloth loaded successfully


In [8]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import (
    TrainingArguments,
    DataCollatorForSeq2Seq,
)
import torch

In [9]:
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.6.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [10]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [11]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 32,
    lora_dropout =  0.05,
    bias = 'none',
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.7 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
dataset = load_dataset("json",
                       data_files = "/home/lakshay/Argus AI/argus/resume/training/resume_adapter_v1_standardized.jsonl",
                       split = 'train'
                      )

split = dataset.train_test_split(
    test_size=0.10,
    seed=42,
)

train_dataset = split["train"]
eval_dataset = split["test"]

In [ ]:
import json

with open(
    "/home/lakshay/Argus AI/argus/resume/training/resume_adapter_v1_standardized.jsonl",
    "r",
    encoding="utf-8"
) as f:

    seen = set()

    for line in f:
        row = json.loads(line)

        if row["instruction"] not in seen:
            seen.add(row["instruction"])

            print("=" * 80)
            print("INSTRUCTION:", row["instruction"])
            print("INPUT:")
            print(row["input"][:500])
            print("OUTPUT:")
            print(row["output"])
            print()

INSTRUCTION: Generate a professional resume summary.
INPUT:
EXPERIENCE
Clinical Coordinator | City General Hospital | Aug 2020 - Present
- Manage shift schedules for 30+ nursing staff, ensuring optimal patient-to-nurse ratios are maintained.
- Assist the department director in auditing patient records for compliance with Joint Commission standards.

Registered Nurse (ICU) | City General Hospital | Jun 2015 - Jul 2020
- Provided direct critical care to patients with complex medical conditions.
- Collaborated with multidisciplinary healthcare teams to deve
OUTPUT:
Healthcare administrator with a strong clinical foundation, offering eight years of combined experience in intensive care nursing and departmental coordination. Expert in managing complex staff schedules for 30+ personnel to ensure optimal operational coverage and patient safety. Demonstrated capability in assisting department directors with rigorous clinical audits to maintain strict Joint Commission compliance. Recently earne

In [14]:
EOS_TOKEN = tokenizer.eos_token

def format_example(example):

    text = (
        "<|begin_of_text|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{example['instruction']}\n\n"
        f"Input:\n{example['input']}\n\n"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"Output:\n{example['output']}\n"
        "<|eot_id|>"
    )

    return {"text": text}

In [15]:
train_dataset = train_dataset.map(format_example)
eval_dataset = eval_dataset.map(format_example)

In [16]:
training_args = TrainingArguments(
    output_dir="resume_adapter_v2",

    num_train_epochs=2,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=1e-4,

    weight_decay=0.01,

    warmup_ratio=0.05,

    lr_scheduler_type="cosine",

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=50,


    save_strategy="no",

    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),

    optim="adamw_8bit",

    report_to="none",

    seed=42,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [17]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    dataset_text_field="text",

    max_seq_length=4096,

    packing=False,

    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        padding=True,
    ),

    args=training_args,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/6481 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/721 [00:00<?, ? examples/s]

In [18]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=6):   0%|          | 0/6481 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/6481 [00:00<?, ? examples/s]

Unsloth: Removed 762 out of 6481 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


Map (num_proc=6):   0%|          | 0/721 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/721 [00:00<?, ? examples/s]

Unsloth: Removed 85 out of 721 samples from eval_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


In [19]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,719 | Num Epochs = 2 | Total steps = 1,430
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 22,544,384 of 1,258,358,784 (1.79% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
50,0.981257,0.746689
100,1.113327,0.583810
150,0.938081,0.548322
200,0.940490,0.503279
250,0.913219,0.475917
300,0.801290,0.445553
350,0.855902,0.424052
400,0.769047,0.420778
450,0.866641,0.438306
500,0.922894,0.429563


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=1430, training_loss=0.7337445042350076, metrics={'train_runtime': 2700.2295, 'train_samples_per_second': 4.236, 'train_steps_per_second': 0.53, 'total_flos': 8196739232219136.0, 'train_loss': 0.7337445042350076, 'epoch': 2.0})

In [34]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

RuntimeError: on_train_begin must be called before on_evaluate

### Extracting Evaluation Metrics from Training History

Since `eval_strategy="steps"` was enabled, the evaluation metrics, including `eval_loss`, were logged during training. We can extract these from `trainer.state.log_history`.

In [36]:
eval_metrics = [log for log in trainer.state.log_history if 'eval_loss' in log]

if eval_metrics:
    latest_eval_loss = eval_metrics[-1]['eval_loss']
    print(f"Latest Evaluation Loss from training history: {latest_eval_loss}")
else:
    print("No evaluation loss found in training history.")

Latest Evaluation Loss from training history: 0.33365485072135925


In [35]:
eval_loss

NameError: name 'eval_loss' is not defined

In [20]:

model.save_pretrained("resume_adapter_v2")
tokenizer.save_pretrained("resume_adapter_v2")

Unsloth: Restored added_tokens_decoder metadata in resume_adapter_v2/tokenizer_config.json.


('resume_adapter_v2/tokenizer_config.json',
 'resume_adapter_v2/chat_template.jinja',
 'resume_adapter_v2/tokenizer.json')

In [21]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (

In [22]:
generation_config = {
    "max_new_tokens": 512,
    "temperature": 0.3,
    "top_p": 0.9,
    "do_sample": True,
    "repetition_penalty": 1.1,
}

In [ ]:
resume_adapter_v2 = "/home/lakshay/Argus AI/argus/resume/adapter/resume_adapter_v2"

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="resume_adapter_v2",
    max_seq_length=4096,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.6.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (

In [32]:
prompt = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Review this resume and provide ATS feedback.

Input:
Senior DevOps Engineer | 6 YOE | Docker | Kubernetes | Terraform

<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""


inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to("cuda")


outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=False,
    )
)

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|begin_of_text|><|begin_of_text|><|start_header_id|>user<|end_header_id|>

Review this resume and provide ATS feedback.

Input:
Senior DevOps Engineer | 6 YOE | Docker | Kubernetes | Terraform

<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Output:
{"ats_score": 88, "strengths": ["Modern containerization (Docker)", "Cloud orchestration (Kubernetes)", "Environment variables management (Terraform)"], "weaknesses": ["No GitOps pipeline experience", "Lacks enterprise DevSecOps integration", "Missing multi-cloud infrastructure design"], "suggestions": ["Architect GitOps pipelines with ArgoCD", "Integrate security protocols (Zero Trust)", "Design multi-cloud architectures with AWS/GCP"], "verdict": "Potential Fit"}
<|eot_id|>
